## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [43]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key



'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'

In [44]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FC6543ED70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FC6543F6F0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [46]:
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
chain=model|parser
chain.invoke([HumanMessage(content="Hi , My name is Kumar and I am a CSE 3rd year Student at RGUKT University. I am interested in learning about AI and its applications. Can you tell me more about it?    ")])
# model.invoke([HumanMessage(content="Hi , My name is Kumar and I am a CSE 3rd year Student at RGUKT University. I am interested in learning about AI and its applications. Can you tell me more about it?    ")])

"Nice to meet you, Kumar. I'd be happy to help you learn more about Artificial Intelligence (AI) and its applications.\n\nArtificial Intelligence is a branch of computer science that focuses on creating intelligent machines that can think, learn, and behave like humans. AI involves the development of algorithms, statistical models, and machine learning techniques to enable computers to perform tasks that would normally require human intelligence.\n\nSome of the key areas of AI include:\n\n1. **Machine Learning**: This is a subset of AI that involves training algorithms to learn from data and make predictions or decisions.\n2. **Deep Learning**: This is a type of machine learning that uses neural networks to analyze data and make decisions.\n3. **Natural Language Processing (NLP)**: This involves enabling computers to understand and generate human language.\n4. **Computer Vision**: This involves enabling computers to interpret and understand visual data from images and videos.\n5. **Rob

In [47]:
from langchain_core.messages import AIMessage
ans=model.invoke(
    [
        HumanMessage(content="Hi , My name is Kumar and I am a CSE 3rd year Student at RGUKT University. I am interested in learning about AI and its applications. Can you tell me more about it?    "),
        AIMessage(content="Hello Kumar! It's nice to meet you. \n\nAs a CSE 3rd year student, you're in an exciting field! AI is rapidly growing and has applications in many areas like healthcare, finance, transportation, and more.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)
print(ans)

content="Your name is Kumar, and you're a 3rd year student studying Computer Science and Engineering (CSE) at RGUKT University. You're also interested in learning about Artificial Intelligence (AI) and its applications." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 147, 'total_tokens': 193, 'completion_time': 0.056826518, 'completion_tokens_details': None, 'prompt_time': 0.009779147, 'prompt_tokens_details': None, 'queue_time': 0.046135163, 'total_time': 0.066605665}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019c5773-bb2e-78b1-aa9d-206669f0cbaf-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 147, 'output_tokens': 46, 'total_tokens': 193}


### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [48]:
!pip install langchain_community

In [49]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [50]:
config={"configurable":{"session_id":"chat1"}}

In [51]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Kumar and I am a CSE 3rd year Student at RGUKT University. I am interested in learning about AI and its applications. Can you tell me more about it?    ")],
    config=config
)

In [52]:
response.content

"Hello Kumar, nice to meet you.  I'd be happy to help you learn more about AI and its applications.\n\nArtificial Intelligence (AI) is a subfield of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence, such as:\n\n1. **Learning**: AI systems can learn from data, experiences, or interactions, and improve their performance over time.\n2. **Problem-solving**: AI systems can solve complex problems, make decisions, and plan actions.\n3. **Perception**: AI systems can perceive and interpret data from sensors, images, and other sources.\n4. **Reasoning**: AI systems can reason and draw conclusions based on data and rules.\n\nThere are many applications of AI in various fields, such as:\n\n1. **Computer Vision**: AI can be used for image recognition, object detection, facial recognition, and more.\n2. **Natural Language Processing (NLP)**: AI can be used for text analysis, sentiment analysis, language translation, and 

In [54]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Kumar.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 913, 'total_tokens': 919, 'completion_time': 0.004486651, 'completion_tokens_details': None, 'prompt_time': 0.064425444, 'prompt_tokens_details': None, 'queue_time': 0.046267616, 'total_time': 0.068912095}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c5775-a4e3-7e92-8545-7a64f23403bb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 913, 'output_tokens': 6, 'total_tokens': 919})

In [55]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"I don't have any information about your name as our conversation just started and I'm a large language model, I don't have personal knowledge or access to any information about individual users. If you'd like to share your name, I can use it to address you in our conversation."

In [56]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

"Hi John, nice to meet you. I'll make sure to address you by your name from now on. How can I assist you today? Do you have a specific question, topic, or issue you'd like to discuss?"

In [57]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [58]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [59]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Kumar")]})

AIMessage(content="Hello Kumar, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 57, 'total_tokens': 83, 'completion_time': 0.063653154, 'completion_tokens_details': None, 'prompt_time': 0.010644176, 'prompt_tokens_details': None, 'queue_time': 0.066916934, 'total_time': 0.07429733}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c5779-6af1-7311-90b1-9bdd6731b5c9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 26, 'total_tokens': 83})

In [60]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [62]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Kumar")],
    config=config
)

response

AIMessage(content="Nice to meet you, Kumar. It seems we've had a similar introduction earlier. How are you doing today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 104, 'total_tokens': 128, 'completion_time': 0.053333095, 'completion_tokens_details': None, 'prompt_time': 0.029906066, 'prompt_tokens_details': None, 'queue_time': 0.046798599, 'total_time': 0.083239161}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c5779-a491-7c93-a476-b464ab68eb17-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 104, 'output_tokens': 24, 'total_tokens': 128})

In [63]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Kumar.'

In [65]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [67]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Kumar")],"language":"telugu"})
response.content

'మీ పేరు కుమార్ అని మీరు చెప్పారు కదా! నేను మీకు సహాయం చేయడానికి ఇక్కడే ఉన్నాను. ఏమంటున్నారు?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [68]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [69]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Kumar")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते कुमार जी, मैं आपकी सहायता के लिए तैयार हूँ। क्या मैं कुछ भी कर सकता हूँ?'

In [70]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [71]:
response.content

'आपका नाम कुमार है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [74]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=30,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm Kumar"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [75]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=100,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm Kumar"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm Kumar", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwar

In [76]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'You like vanilla ice cream.'

In [77]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked me to solve the math problem 2 + 2.'

In [78]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [79]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'Your name is Kumar.'

In [81]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You initially asked the math problem 2 + 2.'